# Task 3 — SmallCNN E3 Experiments

This notebook trains only the two predeclared E3 children. It does **not** retrain E1 or E2.

1. Gender starts from the accepted E1 baseline and changes only to capped class-balanced cross-entropy.
2. Usage starts from the accepted class-balanced E2 model and adds only dropout `p=0.20` between global pooling and the final classifier.

Both models remain the same scratch SmallCNN. Select a Colab GPU runtime, then use Run All.

## 1. Mount Drive and load the submitted branch

Drive supplies the dataset, saved E1/E2 parents, registry, and persistent E3 output.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"


def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

In [2]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)

if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=REPO_DIR, text=True
    ).strip().splitlines()
    if dirty:
        print("Local repository changes found:")
        for change in dirty:
            print(f"  {change}")
        print("Trying a safe fast-forward update. Git will stop before overwriting a local file.")
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print(f"Repository ready: {REPO_DIR}")
print(f"Branch: {BRANCH}")
print(f"Commit: {commit}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin task-3-gender-usage-classification
$ git switch task-3-gender-usage-classification
Local repository changes found:
  ?? results/figures/task3/
Trying a safe fast-forward update. Git will stop before overwriting a local file.
$ git merge --ff-only origin/task-3-gender-usage-classification
Repository ready: /content/MLA2
Branch: task-3-gender-usage-classification
Commit: 8d84d875a75a1692607e23e6a80b6dd48d59f408


## 2. Copy the teacher data onto the runtime disk

Training reads images from Colab's local disk. The archive keeps the repository folder structure.

In [3]:
if not DATA_ZIP.is_file():
    raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}

with zipfile.ZipFile(DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe_names = [name for name in names if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe_names:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in the expected folder.")
    current_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
    needs_extract = current_images != expected_images or not all(path.is_file() for path in required_files)
    if needs_extract:
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(path.suffix.lower() in image_suffixes for path in teacher_dir.rglob("*"))
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")

Teacher data is already extracted; skipping.
Teacher data ready: 44,441 images


## 3. Resolve the saved parents and verify both E3 children

Gender resolves its five E1 baseline folds. Usage resolves its five accepted E2 class-balanced folds. The checks do not take an optimiser step.

In [4]:
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

for output_dir in (DRIVE_TASK_DIR / "experiments", DRIVE_TASK_DIR / "logs", DRIVE_TASK_DIR / "results"):
    output_dir.mkdir(parents=True, exist_ok=True)

from fashion.train.task3_experiments import (
    check_task3_child_setup,
    latest_completed_baseline_parent_run_ids,
    latest_completed_usage_e2_parent_run_ids,
)

gender_parent_run_ids = latest_completed_baseline_parent_run_ids(
    "gender", output_root=DRIVE_TASK_DIR
)
usage_parent_run_ids = latest_completed_usage_e2_parent_run_ids(
    output_root=DRIVE_TASK_DIR
)
gender_e3_check = check_task3_child_setup(
    "gender_class_balanced", parent_run_ids=gender_parent_run_ids, root=REPO_DIR, device_name="cuda"
)
usage_e3_check = check_task3_child_setup(
    "usage_classifier_dropout", parent_run_ids=usage_parent_run_ids, root=REPO_DIR, device_name="cuda"
)

print("GPU:", gender_e3_check["environment"]["gpu"])
print("Gender E1 parents:", gender_parent_run_ids)
print("Usage E2 parents: ", usage_parent_run_ids)
print("Gender E3 change:", gender_e3_check["changed_factor"])
print("Usage E3 change: ", usage_e3_check["changed_factor"])
print("Optimizer steps during checks:", gender_e3_check["optimizer_steps"], usage_e3_check["optimizer_steps"])

GPU: NVIDIA L4
Gender E1 parents: ('t3_baseline_gender_smallcnn_f0_s2753_e46cd00adf0a_20260830T082833Zf8c1e0', 't3_baseline_gender_smallcnn_f1_s2753_e46cd00adf0a_20260830T083708Z143950', 't3_baseline_gender_smallcnn_f2_s2753_e46cd00adf0a_20260830T084548Z5acbf0', 't3_baseline_gender_smallcnn_f3_s2753_e46cd00adf0a_20260830T085427Z5d34f9', 't3_baseline_gender_smallcnn_f4_s2753_e46cd00adf0a_20260830T090303Z41a843')
Usage E2 parents:  ('t3_usage_e2_class_balanced_ce_usage_smallcnn_f0_s2753_5461e048c3b3_20260830T115815Z356f6d', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f1_s2753_5461e048c3b3_20260830T120645Z6d08cd', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f2_s2753_5461e048c3b3_20260830T121514Zd58aa0', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f3_s2753_5461e048c3b3_20260830T122347Z2cb06e', 't3_usage_e2_class_balanced_ce_usage_smallcnn_f4_s2753_5461e048c3b3_20260830T123218Z94db47')
Gender E3 change: class_balanced_loss
Usage E3 change:  classifier_dropout
Optimizer steps during ch

## 4. Train Gender E3: class-balanced loss

This foreground cell trains folds 0–4. Only the loss changes from the matching Gender E1 parent.

In [5]:
from fashion.train.task3_experiments import run_task3_child_cv

gender_e3_result = run_task3_child_cv(
    "gender_class_balanced",
    parent_run_ids=gender_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
gender_e3_result

[task3] starting five-fold experiment=t3_gender_class_balanced_smallcnn for target=gender
[task3] preparing target=gender fold=0: train=26,220, validation=6,553
[task3] fitting fold-training RGB statistics for target=gender fold=0
[task3] RGB statistics ready for target=gender fold=0
[task3] registered t3_gender_e3_class_balanced_ce_gender_smallcnn_f0_s2753_5505bf0b6c74_20260830T143830Z3e324b; the first optimiser step may now run
[task3] target=gender fold=0 epoch=1/30 train_loss=0.7287 train_macro_f1=0.4807 validation_loss=1.0934 validation_macro_f1=0.3726
[task3] target=gender fold=0 epoch=2/30 train_loss=0.5423 train_macro_f1=0.6088 validation_loss=0.5625 validation_macro_f1=0.6130
[task3] target=gender fold=0 epoch=3/30 train_loss=0.4764 train_macro_f1=0.6586 validation_loss=0.8266 validation_macro_f1=0.4873
[task3] target=gender fold=0 epoch=4/30 train_loss=0.4357 train_macro_f1=0.7003 validation_loss=0.4771 validation_macro_f1=0.6631
[task3] target=gender fold=0 epoch=5/30 train_

{'target': 'gender',
 'fold_run_ids': ['t3_gender_e3_class_balanced_ce_gender_smallcnn_f0_s2753_5505bf0b6c74_20260830T143830Z3e324b',
  't3_gender_e3_class_balanced_ce_gender_smallcnn_f1_s2753_5505bf0b6c74_20260830T144710Z7eca8a',
  't3_gender_e3_class_balanced_ce_gender_smallcnn_f2_s2753_5505bf0b6c74_20260830T145555Z083177',
  't3_gender_e3_class_balanced_ce_gender_smallcnn_f3_s2753_5505bf0b6c74_20260830T150442Z0f9ad7',
  't3_gender_e3_class_balanced_ce_gender_smallcnn_f4_s2753_5505bf0b6c74_20260830T151330Z474944'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e3_class_balanced_ce/gender/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e3_class_balanced_ce/gender/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e3_class_balanced_ce/gender/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_gender_e3_c

## 5. Train Usage E3: classifier dropout

This foreground cell trains folds 0–4. It keeps the accepted E2 weighted loss and adds only dropout `p=0.20` before the final classifier.

In [6]:
usage_e3_result = run_task3_child_cv(
    "usage_classifier_dropout",
    parent_run_ids=usage_parent_run_ids,
    folds=range(5),
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[REPO_DIR / "results/runs.csv"],
    device_name="cuda",
)
usage_e3_result

[task3] starting five-fold experiment=t3_usage_classifier_dropout_smallcnn for target=usage
[task3] preparing target=usage fold=0: train=26,219, validation=6,553
[task3] fitting fold-training RGB statistics for target=usage fold=0
[task3] RGB statistics ready for target=usage fold=0
[task3] registered t3_usage_e3_classifier_dropout_usage_smallcnn_f0_s2753_55b7a96c2617_20260830T152222Z130be4; the first optimiser step may now run
[task3] target=usage fold=0 epoch=1/30 train_loss=1.2271 train_macro_f1=0.1591 validation_loss=1.0802 validation_macro_f1=0.1852
[task3] target=usage fold=0 epoch=2/30 train_loss=1.1313 train_macro_f1=0.2428 validation_loss=1.0145 validation_macro_f1=0.2640
[task3] target=usage fold=0 epoch=3/30 train_loss=0.9865 train_macro_f1=0.2905 validation_loss=0.8780 validation_macro_f1=0.2983
[task3] target=usage fold=0 epoch=4/30 train_loss=0.8685 train_macro_f1=0.3044 validation_loss=0.8534 validation_macro_f1=0.3353
[task3] target=usage fold=0 epoch=5/30 train_loss=0.

{'target': 'usage',
 'fold_run_ids': ['t3_usage_e3_classifier_dropout_usage_smallcnn_f0_s2753_55b7a96c2617_20260830T152222Z130be4',
  't3_usage_e3_classifier_dropout_usage_smallcnn_f1_s2753_55b7a96c2617_20260830T153109Ze3f157',
  't3_usage_e3_classifier_dropout_usage_smallcnn_f2_s2753_55b7a96c2617_20260830T153955Z36ee4d',
  't3_usage_e3_classifier_dropout_usage_smallcnn_f3_s2753_55b7a96c2617_20260830T154844Z0738d5',
  't3_usage_e3_classifier_dropout_usage_smallcnn_f4_s2753_55b7a96c2617_20260830T155731Z30497c'],
 'prediction_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e3_classifier_dropout/usage/aggregate/oof_predictions.csv',
 'metrics_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e3_classifier_dropout/usage/aggregate/metrics.json',
 'class_report_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e3_classifier_dropout/usage/aggregate/per_class.csv',
 'confusion_path': '/content/drive/MyDrive/MLA2/task3/experiments/t3_usage_e3_classifier_

## 6. Show the first parent–child comparison

This small table is only a first check. The main Task 3 notebook must still apply every prewritten gate to curves, classes, paired OOF predictions, fold spread, calibration, and robustness.

In [7]:
import pandas as pd

parent_metric_paths = {
    "gender": DRIVE_TASK_DIR / "baseline/gender/aggregate/metrics.json",
    "usage": DRIVE_TASK_DIR / "experiments/t3_usage_e2_class_balanced_ce/usage/aggregate/metrics.json",
}
children = {"gender": gender_e3_result, "usage": usage_e3_result}
comparison = []
for target, child in children.items():
    parent = json.loads(parent_metric_paths[target].read_text(encoding="utf-8"))
    comparison.append({
        "target": target,
        "parent_macro_f1": parent["macro_f1"],
        "e3_macro_f1": child["metrics"]["macro_f1"],
        "macro_f1_change": child["metrics"]["macro_f1"] - parent["macro_f1"],
        "e3_metrics_path": child["metrics_path"],
    })
pd.DataFrame(comparison)

,target,parent_macro_f1,e3_macro_f1,macro_f1_change,e3_metrics_path
0,gender,0.711753,0.708114,-0.003639,/content/drive/MyDrive/MLA2/task3/experiments/...
1,usage,0.408171,0.416133,0.007962,/content/drive/MyDrive/MLA2/task3/experiments/...
